In [ ]:
# -*- coding: utf-8 -*- 
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import VarianceThreshold

# ---------- 0) Load ----------
path = "df_after_age_onehot.csv"
df = pd.read_csv(path, low_memory=False)

target_col = "기부여부"

# ---------- 1) 결측치 50% 이상 컬럼 제거 ----------
df = df.drop(columns=df.columns[df.isnull().mean() >= 0.5])

# ---------- 2) 수치형 / 범주형 분리 후 결측치 채우기 ----------
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

# 수치형 → 중앙값
imputer_num = SimpleImputer(strategy="median")  ### 
df[num_cols] = imputer_num.fit_transform(df[num_cols])

# 범주형 → 최빈값 후 숫자형 변환
imputer_cat = SimpleImputer(strategy="most_frequent")
df[cat_cols] = imputer_cat.fit_transform(df[cat_cols])
for c in cat_cols:
    df[c] = df[c].astype("category").cat.codes  # 숫자형 코드로 변환

# ---------- 3) 타겟 분리 ----------
df[target_col] = df[target_col].astype(int)
X = df.drop(columns=[target_col])
y = df[target_col]

# ---------- 4) 저카디널리티(고유값 ≤ 10) 컬럼만 원핫 인코딩 ----------
low_cardinality_cols = X.columns[X.nunique() <= 10]
X = pd.get_dummies(X, columns=low_cardinality_cols, drop_first=True)

# ---------- 5) 분산이 0.01 이하인 컬럼 제거 (컬럼명 유지) ----------
selector = VarianceThreshold(threshold=0.01)
X_sel = selector.fit_transform(X)
X = pd.DataFrame(X_sel, columns=X.columns[selector.get_support(indices=True)])

# ---------- 6) 다중공선성 제거 (상관계수 > 0.9) ----------
corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
drop_cols = [col for col in upper.columns if any(upper[col] > 0.9)]
X = X.drop(columns=drop_cols)

# ---------- 7) 여러 버전 생성 ----------
X_raw = X.copy()

scaler_std = StandardScaler()
X_std = pd.DataFrame(scaler_std.fit_transform(X), columns=X.columns)

scaler_mm = MinMaxScaler()
X_mm = pd.DataFrame(scaler_mm.fit_transform(X), columns=X.columns)

print("Shapes")
print("Raw:", X_raw.shape, "Std:", X_std.shape, "MinMax:", X_mm.shape, "Target:", y.shape)

Shapes
Raw: (184915, 460) Std: (184915, 460) MinMax: (184915, 460) Target: (184915,)


In [ ]:
from sklearn.feature_selection import VarianceThreshold

# ---------- 분산 계산 ----------
variances = X.var()  # 각 컬럼별 분산
low_variance_cols = variances[variances <= 0.01].index.tolist()

print("📉 분산이 0.01 이하인 컬럼 개수:", len(low_variance_cols))
print("🗑 컬럼 목록:", low_variance_cols)

# (선택) 분산 값까지 보고 싶다면
low_variance_report = variances[variances <= 0.01].sort_values()
print("\n📊 저분산 컬럼 요약:")
print(low_variance_report)


📉 분산이 0.01 이하인 컬럼 개수: 0
🗑 컬럼 목록: []

📊 저분산 컬럼 요약:
Series([], dtype: float64)


In [ ]:
# ---------- 8) 결과 저장 ----------
X_raw.to_csv("X_raw.csv", index=False, encoding="utf-8-sig")
X_std.to_csv("X_std.csv", index=False, encoding="utf-8-sig")
X_mm.to_csv("X_mm.csv", index=False, encoding="utf-8-sig")
y.to_csv("y.csv", index=False, encoding="utf-8-sig")

print("저장 완료")
print(" - X_raw.csv:", X_raw.shape)
print(" - X_std.csv:", X_std.shape)
print(" - X_mm.csv:", X_mm.shape)
print(" - y.csv:", y.shape)

📂 저장 완료
 - X_raw.csv: (184915, 460)
 - X_std.csv: (184915, 460)
 - X_mm.csv: (184915, 460)
 - y.csv: (184915,)
